# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib
import os
from dotenv import load_dotenv
import isodate
import datetime

def get_isodatetime():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.datetime_isoformat(current)
    return iso_value

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value
#local functions
import video_extraction
import audio_transcription
import transcript_merger
import summarize_transcript

# Load environment variables
load_dotenv()
logger = video_extraction.logger

## Transcribe

In [ ]:
# Args
#-- extraction
dir_subfolder="conductor" #change for subfolder export. "" is none
input_mkv = "./+recordings/2024-07-25_10-32-44-meetingnotes.mkv" 

# -- transcribing
mic_audio_filename = f"microphone_audio.wav"
desktop_audio_filename = f"desktop_audio.wav"
# -- processing/merging
mic_transcript_filename = f"microphone_transcription.txt"
desktop_transcript_filename = f"desktop_transcription.txt"
transcript_filename = f"merged_transcript.json"
# -- summarizing
summary_filename=f"summary.txt"
summarize_model_name = 'claude-3-5-sonnet'

# Script Code
# - Setup output directories
output_dir = f"./{get_isodate()}-output"
output_path = pathlib.Path(output_dir) if dir_subfolder in ["",None,False] else pathlib.Path(output_dir) / dir_subfolder
if not output_path.exists():
    output_path.mkdir(parents=True)
    
print(f"Output dir is: {output_path}")
# -- Extract audio files
try:
   result = video_extraction.process_mkv_file(input_mkv, str(output_path))
except Exception as e:
    logger.error(f"An error occurred during the extraction process: {str(e)}")
    
# -- Transcribe audio
mic_audio_path = output_path / mic_audio_filename
desktop_audio_path = output_path / desktop_audio_filename
try:
    filenames = audio_transcription.process_audio_files(str(mic_audio_path), str(desktop_audio_path), output_dir=str(output_path))
    logger.info(filenames)
except Exception as e:
    logger.error(f"An error occurred during the transcription process: {str(e)}")

# -- Merge and process transcription
mic_transcript_path = filenames['microphone']
desktop_transcript_path = filenames['desktop']
transcript_file_path = output_path / transcript_filename
try:
    transcript_json = transcript_merger.process_transcripts(
        mic_transcript_file=str(mic_transcript_path), 
        desktop_transcript_file=str(desktop_transcript_path), 
        output_file=str(transcript_file_path),
        parse_overlaps=False
    )
except Exception as e:
    logger.error(f"An error occurred during the merging process: {str(e)}")

# -- Augment with additional information
# TODO: add personal notes taken during meeting
# TODO: add other md, txt, or other text files


# -- Summarize
# Main workflow
summary_filepath = output_path / summary_filename

transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=summarize_model_name
)

# summarize=False #True to call the llm
# # if summarize:
# #     try:
#         summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name)
#         summarize_transcript.save_summary(
#             summary, 
#             output_file=str(summary_filepath)
#         )
#         logger.info(f"Summary generated and saved to {summary_filepath}")
#     except Exception as e:
#         logger.error(f"An error occurred during summarization: {str(e)}")
#     logger.info(summary)
# else:
#     logger.info("Skipping summarize step")

## Summarize

In [ ]:
import summarize_transcript
from anthropic import Anthropic

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value

def get_filename_prompt(content):
    current_date = get_isodate()
    prompt = f"""Please read the <content> compeltely and provide a filename that can be used to write the file to disk as a markdown. The filename should be concise but also descriptive so its clear what the content contains.

    <filenamerules>
    - only use lowercase letters, numbers, and underscores or dashes
    - prefer underscores where possible as they are operating system independent
    - make the filename convenient for all operating systems (windows, macOS, linux)
    - include the current date in the filename as YYYMMDD (e.g. July 4th , 2024 would be 20240704)
    - include a concise but descriptive name related to the <content>
    - keep the filename as short as possible while containing a useful name and date
    - DONOT include the suffix of the filename and make sure their are no period characters in the filename
    - the current date is {current_date}
    </filenamerules>

    <content>
    {content}
    </content>

    remember ONLY return the filename as a string and follow the <filenamerules>
    """
    return prompt
     

def get_filename_from_content(content, model_name='claude-3-5-sonnet'):
    """obtain a good filename from content and date"""
    
    anthropic_models = summarize_transcript.anthropic_models
    if model_name not in anthropic_models:
        raise KeyError(f"model must be one of [{list(anthropic_models.keys())}]")
    model = anthropic_models.get(model_name)['name']
    client = Anthropic(api_key=summarize_transcript.ANTHROPIC_API_KEY)
    prompt = get_filename_prompt(content)
    
    try:
        response = client.messages.create(
            model=model,
            max_tokens=100,
            temperature=0.7,
            messages=[
                {"role": "user", "content": prompt}
            ],
        )
        return response.content[0].text
    except Exception as e:
        logger.error(f"Error generating summary: {str(e)}")
        raise

def create_markdown_report(content, output_path):
    """reads content and comes up with a good name, then writes to markdown"""
    filename = get_filename_from_content(content, model_name='claude-3-5-sonnet')
    filepath = output_path / f"{filename}.md"
    logger.info(f"Writing markdown file to {filepath}")
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(content)
    return filepath


In [ ]:
summarize = True

# -- Summarize
# Main workflow
summary_filepath = output_path / f"{get_isodatetime()}-{summary_filename}"
transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=summarize_model_name
)
logger.info(f"Summary filename: {summary_filepath}")
if summarize:
    try:
        summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name)
        summarize_transcript.save_summary(
            summary, 
            output_file=str(summary_filepath)
        )
        logger.info(f"Summary generated and saved to {summary_filepath}")
        filepath = create_markdown_report(content=summary, output_path=output_path)
    except Exception as e:
        logger.error(f"An error occurred during summarization: {str(e)}")
else:
    logger.info("Skipping summarize step")
